# Project 5:  Bayesian and Imputation Analysis

## Reminder: US Accidents (New York Subset) Dataset Overview

[Our dataset](https://docs.google.com/spreadsheets/d/1KY1tcftko3W9Fp-ZbceReVZD2G58tGEhJw6xMqyzIDw/edit?usp=sharing) (generated in ProjEx1) is a processed subset of the nationwide US Accidents (2016–2023) [dataset](https://www.kaggle.com/datasets/sobhanmoosavi/us-accidents), filtered for incidents occurring in NYC ($N=18,883$ after cleaning).

| Var | Type | Data Type | Description |
| --- | --- | --- | --- |
| **`accident_id`** | Categorical | `int64` | Unique numeric identifier extracted from the original ID. |
| **`severity_of_traffic`** | Ordinal | `int8` | Severity score **1–4** (1 = lowest impact, 4 = highest impact). |
| **`traffic_disruption_duration`** | Continuous | `float64` | Duration of the traffic impact in **hours**. |
| **`distance_of_traffic`** | Continuous | `float64` | Distance in **miles** over which traffic flow was affected. |
| **`is_day`** | Binary | `int8` | **1** = Day, **0** = Night (derived from Sunrise/Sunset). |
| **`is_traffic_signal`** | Binary | `int8` | **1** = Yes, **0** = No (presence of a traffic signal nearby). |

The dataset focuses on traffic impact, where both duration and distance exhibit extreme right-skewness. `traffic_disruption_duration` contains massive outliers (up to ~3,967h) despite a low median (1.26h), while `distance_of_traffic` is also zero-inflated.

### Importing the data

In [1]:
import pandas as pd

url = "https://docs.google.com/spreadsheets/d/1mJ3MtFkxHFdTEki1HjrdkAfIvekmWvltFn5S1KRnna8/export?format=csv"
df = pd.read_csv(url)
df.head()

,accident_id,severity_of_traffic,traffic_disruption_duration,distance_of_traffic,is_day,is_traffic_signal
0,194408,2,0.493611,0.01,1,1
1,194422,3,0.494167,1.13,1,0
2,194475,2,0.491389,0.01,1,0
3,194540,2,0.497500,0.01,0,1
4,194558,2,0.492222,0.01,1,1


## Part 1: Bayesian Approach
### Recap from Task 2
**Research question:** Is the distribution of `traffic_disruption_duration` different between accidents that occur with a nearby traffic signal and those without (`is_traffic_signal` is `0` or `1`)?

- $X=$ `traffic_disruption_duration`, a continuous variable representing the length of time traffic was disrupted in hours.
- $Y=$ `is_traffic_signal`, a binary variable indicating whether there is a traffic signal near the location (`1` = yes, `0` = no).

### Question 1: Sampling 

In [ ]:
import numpy as np

np.random.seed(1997) # reproducibilty 
df_observed = df.sample(n=200)
df_remaining = df.drop(df_observed.index) # disjoint
df_past = df_remaining.sample(n=1000)

Total Data: 18883 samples
Observed Data (Likelihood): 200 samples
Past Data (Prior): 1000 samples
Overlap between sets: 0


### Question 2

#### section (a)

### **Mathematical Formulation**
Define:
$$
Z = \begin{cases}
1 & X > \tau \\
0 & X \le \tau
\end{cases}
$$
>
Let $Y$ be the binary variable denoting the presence of a traffic signal ($Y=1$ for signal, $Y=0$ for no signal). We define the probability of high traffic disruption ($Z=1$) conditional on the group $Y=j$ (where $j \in \{0, 1\}$) as:
>
$$
p_j = P(Z=1 | Y=j)
$$
>
Our objective is to estimate the log odds ratio, $\psi$, defined as:
>
$$
\psi = \eta(p_0) - \eta(p_1)
$$
>
where $\eta(p)$ is the logit function:
>
$$
\eta(p) = \log\left(\frac{p}{1-p}\right)
$$

In [7]:
import pandas as pd
import numpy as np


# Dichotomization
tau = df_observed['traffic_disruption_duration'].median()
df_observed['Z'] = (df_observed['traffic_disruption_duration'] > tau).astype(int)

def get_log_odds_ratio(data):
    # Group 1: No Traffic Signal (Y=0)
    # Group 2: Traffic Signal (Y=1)
    
    p0 = data[data['is_traffic_signal'] == 0]['Z'].mean()
    p1 = data[data['is_traffic_signal'] == 1]['Z'].mean()

    # Log Odds Ratio: psi = eta1 - eta2
    return np.log(p0 / (1 - p0)) -np.log(p1 / (1 - p1))
psi_hat = get_log_odds_ratio(df_observed)

B = 500 
boot_estimates = []

for i in range(B):
    # Resample with replacement
    boot_sample = df_observed.sample(n=len(df_observed), replace=True)
    boot_estimates.append(get_log_odds_ratio(boot_sample))

boot_estimates = np.array(boot_estimates)
ci_lower = np.percentile(boot_estimates, 2.5)
ci_upper = np.percentile(boot_estimates, 97.5)

# Output Results
print(f"Threshold (Median): {tau:.4f} hours")
print(f"Point Estimate (Psi): {psi_hat:.4f}")
print(f"95% Bootstrap CI: [{ci_lower:.4f}, {ci_upper:.4f}]")

Threshold (Median): 1.3414 hours
Point Estimate (Psi): 0.5910
95% Bootstrap CI: [-0.0515, 1.2713]


A positive value indicates that the odds of a "High Duration" accident are higher when there is no traffic signal compared to when there is one. (about 1.8 times higher without a signal in this sample) Since the interval includes 0, we do not have sufficient evidence from this sample of 200 (with 5% significance level) to conclude that the distribution of duration (specifically, the probability of being above the median) differs significantly between accidents with and without traffic signals. The direction of the effect suggests signals might reduce duration (lower odds of high duration).

#### section (B)
The postirior for $p_1$ and $p_2$ is beta dist:

**Uniform Prior:**
$$
p_j \mid \text{data} \sim \text{Beta}(1 + s_j, 1 + n_j - s_j)
$$

**Jeffreys Prior: (as shown in lecture)**
$$
p_j \mid \text{data} \sim \text{Beta}(0.5 + s_j, 0.5 + n_j - s_j)
$$

**Informative Prior**

We start with Jeffreys Prior for the unknown probability $p$:
$$
\pi(p) \propto p^{-1/2}(1-p)^{-1/2} \equiv \text{Beta}(0.5, 0.5)
$$

Given the past data with $n_{past}$ trials and $S_{past}$ successes, the posterior distribution for the past data is:
$$
p \mid \text{data}_{past} \sim \text{Beta}\left(S_{past} + 0.5, \; n_{past} - S_{past} + 0.5\right)
$$

We use the posterior from the past data (Step 1) as the **informative prior** for our current experiment.
$$
\text{Prior}_{new} \equiv \text{Posterior}_{past} = \text{Beta}(\alpha_{prior}, \beta_{prior})
$$
Given the current observed data with $n_{obs}$ trials and $S_{obs}$:
$$
p \mid \text{data}_{obs}, \text{data}_{past} \sim \text{Beta}\left(\alpha_{prior} + S_{obs}, \; \beta_{prior} + n_{obs} - S_{obs}\right)
$$

So we get:
$$
p_j \sim \text{Beta}\left(S_{past,j} + S_{obs,j} + 0.5, \; n_{past,j} + n_{obs,j} - S_{past,j} - S_{obs,j} + 0.5\right)
$$
We use the 1,000 past accidents to set our baseline belief ($\alpha_{past}, \beta_{past}$).


In [25]:
df_past['Z'] = (df_past['traffic_disruption_duration'] > tau).astype(int) 

# Numpy seeds for reproducibility
np.random.seed(42)

def get_counts(data):
    """Returns (successes, trials) for both groups: 
    is_traffic_signal=0 and is_traffic_signal=1"""
    
    g0 = data[data['is_traffic_signal'] == 0]['Z']
    g1 = data[data['is_traffic_signal'] == 1]['Z']
    return (g0.sum(), len(g0)), (g1.sum(), len(g1))

(s0_obs, n0_obs), (s1_obs, n1_obs) = get_counts(df_observed)
(s0_past, n0_past), (s1_past, n1_past) = get_counts(df_past)

# Simulation Function for Bayesian Credible Intervals
def get_bayes_results(alpha0, beta0, alpha1, beta1, n_sim=B):
    # Simulate posteriors for p0 and p1
    rv_p0 = np.random.beta(alpha0, beta0, n_sim)
    rv_p1 = np.random.beta(alpha1, beta1, n_sim)
    
    # Calculate Log Odds Ratio: psi = log(p0/(1-p0)) - log(p1/(1-p1))
    eta0 = np.log(rv_p0 / (1 - rv_p0))
    eta1 = np.log(rv_p1 / (1 - rv_p1))
    psi_sim = eta0 - eta1
    
    # 95% Credible Interval
    ci_low = np.percentile(psi_sim, 2.5)
    ci_high = np.percentile(psi_sim, 97.5)
    return np.mean(psi_sim), ci_low, ci_high

results = {}

# --- Section B: Uniform Prior ---
# Prior Beta(1, 1) adds 1 to s and 1 to (n-s)
results['Uniform'] = get_bayes_results(
    alpha0 = 1 + s0_obs, beta0 = 1 + n0_obs - s0_obs,
    alpha1 = 1 + s1_obs, beta1 = 1 + n1_obs - s1_obs
)

# --- Section C: Jeffreys Prior ---
# Prior Beta(0.5, 0.5) adds 0.5 to s and 0.5 to (n-s)
results['Jeffreys'] = get_bayes_results(
    alpha0 = 0.5 + s0_obs, beta0 = 0.5 + n0_obs - s0_obs,
    alpha1 = 0.5 + s1_obs, beta1 = 0.5 + n1_obs - s1_obs
    
)

# --- Section D: Informative Prior (Past Data) ---

# Step 1: Posterior from Past Data (which becomes Prior for Current)
# alpha = 0.5 + successes_past
# beta  = 0.5 + failures_past
prior_a0 = 0.5 + s0_past
prior_b0 = 0.5 + n0_past - s0_past
prior_a1 = 0.5 + s1_past
prior_b1 = 0.5 + n1_past - s1_past

# Step 2: Combine with observed data
results['Informative'] = get_bayes_results(
    alpha0 = prior_a0 + s0_obs, beta0 = prior_b0 + n0_obs - s0_obs,
    alpha1 = prior_a1 + s1_obs, beta1 = prior_b1 + n1_obs - s1_obs
)

# Output
results['Bootstrap'] = (psi_hat, ci_lower, ci_upper)

print(f"{'Method':<15} {'Estimate':<10} {'95% CI':<15} {'CI Length'}")
print("-" * 60)
for method, res in results.items():
    ci_length = res[2] - res[1]
    print(f"{method:<15} {res[0]:.4f}     [{res[1]:.4f}, {res[2]:.4f}] {ci_length:.4f}")

Method          Estimate   95% CI          CI Length
------------------------------------------------------------
Uniform         0.5784     [-0.1027, 1.2416] 1.3444
Jeffreys        0.6231     [-0.0460, 1.3720] 1.4179
Informative     0.1505     [-0.0942, 0.4076] 0.5018
Bootstrap       0.5910     [-0.0515, 1.2713] 1.3229


All methods indicate that there is no significant difference in traffic disruption duration between accidents with and without traffic signals. (wide confidence intervals that included zero). Maybe due to the small sample size (n=200). Using historical data made the CI smaller closer to zero. Past evidence supports a null hypothesis, instead of the small positive in the smaller observed sample.

## Part 2: Data Imputation



**Research question:** We aim to compare different methods for handling missing data when estimating the linear relationship between traffic disruption duration and various traffic/environmental factors.

* $X_1=$ `is_traffic_signal`, a binary variable (`1` = Yes, `0` = No).
* $X_2=$ `is_day`, a binary variable (`1` = Day, `0` = Night).
* $X_3=$ `distance_of_traffic`, a continuous variable representing the length of the jam in miles.
* $Y=$ `traffic_disruption_duration`, a continuous variable representing the duration of the traffic jam in hours.

We assume the following linear model:

$$
Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \beta_3 X_3 + \epsilon
$$

where we aim to estimate the coefficients $\beta$ despite artificially introduced missing data in $Y$.

### Section (1-2)

In [28]:
import statsmodels.api as sm

np.random.seed(42)

# =========
# Section A
# =========
df_full = df.sample(n=1000).copy()

# =========
# Section B: linear regression on full data
# =========

# Data prep
y_col = 'traffic_disruption_duration'
x_cols = ['is_traffic_signal', 'is_day', 'distance_of_traffic']
X_full = sm.add_constant(df_full[x_cols])
y_full = df_full[y_col]
model_full = sm.OLS(y_full, X_full).fit()

# Calculate Coefficients and 95% Confidence Intervals
# statsmodels uses the t-distribution and covariance matrix automatically
results_full = model_full.conf_int(alpha=0.05)
results_full.columns = ['CI_Lower', 'CI_Upper']
results_full['Coef'] = model_full.params


# Output
print(results_full)


                     CI_Lower  CI_Upper      Coef
const                1.306388  2.066819  1.686603
is_traffic_signal   -0.213348  0.739308  0.262980
is_day              -0.418065  0.408791 -0.004637
distance_of_traffic  0.835539  1.333488  1.084514


### Section (3): value removal

In [40]:
# =========
# Section C: Missing Data 
# =========
np.random.seed(42)

# sort
df_sorted = df_full.sort_values(by=y_col)

# Probabilities linearly increase from 0.2 to 0.8
n = len(df_sorted)
p_missing = np.linspace(0.2, 0.8, n)
R_indicators = np.random.binomial(n=1, p=p_missing)

# removal
df_miss = df_sorted.copy()
df_miss.loc[R_indicators == 1, y_col] = np.nan

print(f"Total Rows: {n}")
print(f"Missing Values Generated: {df_miss[y_col].isna().sum()}")

Total Rows: 1000
Missing Values Generated: 526


### Section (4): linear regerssion with imputation

In [41]:
# part A 
# ======

df_cc = df_miss.dropna(subset=['traffic_disruption_duration'])
X_cc = sm.add_constant(df_cc[['is_traffic_signal', 'is_day', 'distance_of_traffic']])
y_cc = df_cc['traffic_disruption_duration']

# 3. Fit OLS Model
model_cc = sm.OLS(y_cc, X_cc).fit()

# 4. Get Coefficients and Confidence Intervals
results_cc = model_cc.conf_int(alpha=0.05)
results_cc.columns = ['CI_Lower', 'CI_Upper']
results_cc['Coef'] = model_cc.params

print(f"Observations remaining: {len(df_cc)}")
print(results_cc)


Observations remaining: 474
                     CI_Lower  CI_Upper      Coef
const                0.952673  1.802395  1.377534
is_traffic_signal   -0.751003  0.316979 -0.217012
is_day              -0.675800  0.262677 -0.206562
distance_of_traffic  0.317264  0.910960  0.614112


In [46]:
# part B Regression Imputation
# ======
missing_mask = df_miss['traffic_disruption_duration'].isna()
X_missing = sm.add_constant(df_miss.loc[missing_mask, ['is_traffic_signal', 'is_day', 'distance_of_traffic']])

# fill missing values
predicted_y = model_cc.predict(X_missing)
df_imp = df_miss.copy()
df_imp.loc[missing_mask, 'traffic_disruption_duration'] = predicted_y

# Re-run Regression on the full (imputed) dataset
X_imp = sm.add_constant(df_imp[['is_traffic_signal', 'is_day', 'distance_of_traffic']])
y_imp = df_imp['traffic_disruption_duration']
model_imp = sm.OLS(y_imp, X_imp).fit()

# Get Results
results_imp = model_imp.conf_int(alpha=0.05)
results_imp.columns = ['CI_Lower', 'CI_Upper']
results_imp['Coef'] = model_imp.params

print("--- Regression Imputation Results ---")
print(results_imp)

# previous results
print("---  previous Results ---")
print(results_cc)


--- Regression Imputation Results ---
                     CI_Lower  CI_Upper      Coef
const                1.172387  1.582680  1.377534
is_traffic_signal   -0.474016  0.039992 -0.217012
is_day              -0.429628  0.016504 -0.206562
distance_of_traffic  0.479777  0.748447  0.614112
---  previous Results ---
                     CI_Lower  CI_Upper      Coef
const                0.952673  1.802395  1.377534
is_traffic_signal   -0.751003  0.316979 -0.217012
is_day              -0.675800  0.262677 -0.206562
distance_of_traffic  0.317264  0.910960  0.614112


Coefficients: No differnce, Regression imputation fills the missing Y values using the prediction​. These new points lie perfectly on the regression plane. Adding points that fit the model perfectly does not tilt the plane, so the coefficients do not change.

CIs: approximately half smaller: We imputed "perfect" data with zero error (bigger dataset) artificially lowering the estimated variance.

this implies Regression imputation is worse here. (same coeffecients, higher flase precision)

### Section (c+d)

To calculate the standard error for our estimates, we combine the results from our $M=20$ imputed models using the variance decomposition formula (as referenced in Slide 72).


In [47]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# Re-running the MI setup from Section 4c to get the lists needed for 4d
M = 20
beta_list = []
variance_list = []

# Parameters for noise (from Complete Case model)
sigma2 = model_cc.mse_resid 
mu_missing = model_cc.predict(X_missing)

np.random.seed(42)

for m in range(M):
    # 1. Generate noise and impute
    noise = np.random.normal(loc=0, scale=np.sqrt(sigma2), size=len(mu_missing))
    df_imputed = df_miss.copy()
    df_imputed.loc[missing_mask, 'traffic_disruption_duration'] = mu_missing + noise
    
    # 2. Fit Model
    X_m = sm.add_constant(df_imputed[x_cols])
    y_m = df_imputed[y_col]
    model_m = sm.OLS(y_m, X_m).fit()
    
    # 3. Store Coefficients (Beta) and their Variances (Diagonal of Cov matrix)
    beta_list.append(model_m.params.values)
    variance_list.append(np.diag(model_m.cov_params()))

# --- Section 4d: Applying the Formula from Slide 72 ---

# Convert lists to numpy arrays for easy math
betas = np.array(beta_list)       # Shape: (M, 4)
variances = np.array(variance_list) # Shape: (M, 4)

# 1. Calculate Average Coefficient (The Point Estimate)
beta_bar = betas.mean(axis=0)

# 2. Calculate "Within Variance" (Average of the variances from the models)
# This replaces the 1/nI(theta) term as per instructions
var_within = variances.mean(axis=0)

# 3. Calculate "Between Variance" (Variance of the coefficients themselves)
# ddof=1 for sample variance
var_between = betas.var(axis=0, ddof=1)

# 4. Calculate Total Variance (Slide 72 Formula)
# Total = Within + (1 + 1/M) * Between
var_total = var_within + (1 + (1/M)) * var_between

# 5. Calculate SE (Square root of Total Variance)
se_total = np.sqrt(var_total)

# 6. Calculate 95% Confidence Interval
# CI = Estimate +/- 1.96 * SE
ci_lower = beta_bar - 1.96 * se_total
ci_upper = beta_bar + 1.96 * se_total

# Organize Results
param_names = ['Intercept', 'is_traffic_signal', 'is_day', 'distance_of_traffic']
results_mi = pd.DataFrame({
    'Estimate': beta_bar,
    'SE': se_total,
    'CI_Lower': ci_lower,
    'CI_Upper': ci_upper
}, index=param_names)

print("--- Section 4d: Multiple Imputation Results ---")
print(results_mi)

--- Section 4d: Multiple Imputation Results ---
                     Estimate        SE  CI_Lower  CI_Upper
Intercept            1.379976  0.174511  1.037935  1.722017
is_traffic_signal   -0.244493  0.242618 -0.720023  0.231038
is_day              -0.202077  0.191271 -0.576967  0.172814
distance_of_traffic  0.610330  0.132953  0.349743  0.870918


### Section (e-g)

In [50]:
import pandas as pd
import numpy as np
import statsmodels.api as sm


# --- Section 4e: Propensity Score (Logistic Regression) ---
# 1. Create Indicator: R=1 (Observed), R=0 (Missing)
df_miss['R'] = df_miss['traffic_disruption_duration'].notna().astype(int)

# 2. Fit Logistic Regression (Predicting R using X)
X_logit = sm.add_constant(df_miss[['is_traffic_signal', 'is_day', 'distance_of_traffic']])
y_logit = df_miss['R']

logit_model = sm.Logit(y_logit, X_logit).fit(disp=0)

# 3. Calculate Propensity Scores (pi)
# The probability of being observed for EACH row
propensity_scores = logit_model.predict(X_logit)

# --- Section 4f: IPW Estimator (WLS) ---
# 1. Calculate Weights (Inverse Probability)
weights = 1.0 / propensity_scores

# 2. Filter for Observed Data Only
# We run the regression only on what we see, but with weights
mask_obs = df_miss['R'] == 1
df_obs = df_miss[mask_obs]
X_obs = sm.add_constant(df_obs[['is_traffic_signal', 'is_day', 'distance_of_traffic']])
y_obs = df_obs['traffic_disruption_duration']
weights_obs = weights[mask_obs]

# 3. Fit Weighted Least Squares
model_ipw = sm.WLS(y_obs, X_obs, weights=weights_obs).fit()

# --- Section 4g: Bootstrap Confidence Intervals ---
B = 1000
beta_boot_list = []

np.random.seed(42)
for b in range(B):
    # Resample the original data (including missing rows)
    df_boot = df_miss.sample(n=len(df_miss), replace=True)
    
    # Re-estimate Weights (Step 1)
    # We must capture the uncertainty of the weight estimation itself
    try:
        X_b_logit = sm.add_constant(df_boot[['is_traffic_signal', 'is_day', 'distance_of_traffic']])
        logit_b = sm.Logit(df_boot['R'], X_b_logit).fit(disp=0)
        pi_b = logit_b.predict(X_b_logit)
        w_b = 1.0 / pi_b
        
        # Re-estimate Coefficients (Step 2)
        mask_b = df_boot['R'] == 1
        model_b = sm.WLS(df_boot.loc[mask_b, 'traffic_disruption_duration'], 
                         sm.add_constant(df_boot.loc[mask_b, ['is_traffic_signal', 'is_day', 'distance_of_traffic']]),
                         weights=w_b.loc[mask_b]).fit()
        
        beta_boot_list.append(model_b.params.values)
    except:
        continue # Skip failed convergences

# Calculate CI from Bootstrap Distribution
betas_boot = np.array(beta_boot_list)
ci_lower = np.percentile(betas_boot, 2.5, axis=0)
ci_upper = np.percentile(betas_boot, 97.5, axis=0)

# Organize Results
results_ipw = pd.DataFrame({
    'Coef': model_ipw.params,
    'CI_Lower': ci_lower,
    'CI_Upper': ci_upper
})

print("--- Section 4f/g: IPW Results ---")
print(results_ipw)

print(results_full)

--- Section 4f/g: IPW Results ---
                         Coef  CI_Lower  CI_Upper
const                1.373558  0.854746  1.972641
is_traffic_signal   -0.230786 -0.552847  0.084213
is_day              -0.166592 -0.876913  0.363583
distance_of_traffic  0.574934  0.196526  1.124605
                     CI_Lower  CI_Upper      Coef
const                1.306388  2.066819  1.686603
is_traffic_signal   -0.213348  0.739308  0.262980
is_day              -0.418065  0.408791 -0.004637
distance_of_traffic  0.835539  1.333488  1.084514


### Section (h)
The Inverse Probability Weighting (IPW) method failed to recover the true relationship between traffic distance and disruption duration. The observed covariates were insufficient to accurately model the propensity scores.